In [50]:
from uuid import uuid4
from pprint import pprint
from edmlib import EDM_ProvidedCHO, ORE_Aggregation, EDM_Record, Ref, Lit
from pydantic import ValidationError

# Mapping / Building

## Typical usage within an ETL-pipeline @ Kulturpool for an API with custom logic ...

```python
def parse_cho(doc: dict, cho_id: str, mmt: MultimediaType) -> EDM_ProvidedCHO:
    return EDM_ProvidedCHO(
        id=Ref(value=cho_id),
        dcterms_extent=extract_lang_duos_from_groups(doc, "ObjDimensionGrp", "ObjDimensionTxt"),
        dc_identifier=[Lit(value=doc["ObjObjectNumberTxt"])],
        dc_title=[Lit(value=doc["ObjTitleTxt_de"], lang="de")],
        dc_type=extract_lang_duos_from_groups(doc, "ObjClassificationVoc", "LabelTxt"),
        dcterms_isPartOf=extract_lang_duo(doc, "ObjCollectionTxt"),
        dc_subject=extract_lang_duos_from_groups(doc, "ObjKeyWordGrp", "KeyWordVoc"),
        dcterms_medium=extract_lang_duo(doc, "ObjMaterialTxt"),
        dcterms_spatial=extract_lang_duos_from_groups(doc, "ObjGeograficGrp", "ObjPlaceTxt"),
        dc_language=[Lit(value="de")] if mmt == MultimediaType.PDF else None,
        dcterms_temporal=[Lit(value=doc["ObjDateTxt"], lang="de")],
        dc_creator=extract_dc_creator(doc),
        edm_type=declare_edm_type(mmt),
    )
```
- The API delivers JSON and the developer knows (usually due to a mapping table) which custom metadata field is to be mapped to EDM fields.
- Field by field an `EDM_ProvidedCHO` instance is declared.
- Note that `extract_lang_duos_from_groups` is a custom function with logic explicitly written for the API returning a `MixedValuesList`.

```python
def parse_aggregation(doc: dict, cho_id: str, mmt: MultimediaType) -> ORE_Aggregation:
    return ORE_Aggregation(
        id=Ref(value=doc["ObjURLTxt"]),
        edm_aggregatedCHO=Ref(value=cho_id),
        edm_dataProvider=Lit(value="Universität für angewandte Kunst Wien"),
        edm_hasView=extract_edm_has_view(mmt, doc),
        edm_isShownAt=Ref(value=doc["ObjURLTxt"]),
        edm_isShownBy=extract_edm_is_shown_by(mmt, doc),
        edm_object=extract_edm_object(mmt, doc),
        edm_provider=Lit(value="Kulturpool"),
        edm_rights=Ref(value=doc["ObjMulCCLizVoc"]),
    )
```
- Note that `extract_edm_has_view`, `extract_edm_is_shown_by` and `extract_edm_object` are custom functions returning either a `Ref` or `list[Ref]` respectively.

```python
def parse_record(doc: dict) -> EDM_Record:
    identifier = str(uuid4())
    mmt = read_multimedia_type(doc)

    return EDM_Record(
        provided_cho=parse_cho(doc, identifier, mmt),
        aggregation=parse_aggregation(doc, identifier, mmt),
    )
```

## Building a record from scratch

Example:
- title: Bronze Decorative Hatchet (en), Bronzezierbeil (de)
- identifier: 1234
- subject: Praehistory (en), War (https://www.wikidata.org/wiki/Q198) 
- object type: Zierwaffe (de), Weapon (https://www.wikidata.org/wiki/Q728)
- medium: Metall / Bronze (de)
- spatial: Hallstatt (de, en)
- temporal: 800-450 BC

In [51]:
cho_id = str(uuid4())

In [52]:
identifiers = [Lit(value="1234")]
titles = [Lit(value="Bronze Ornamental Hatchet", lang="en"), Lit(value="Bronzezierbeil", lang="de")]
subjects = [Lit(value="Praehistory", lang="en"), Ref(value="https://www.wikidata.org/wiki/Q198")]
object_types = [Lit(value="Decorative Weapon"), Ref(value="https://www.wikidata.org/wiki/Q728")]
media = [Lit(value="Metall", lang="de"), Lit(value="Bronze", lang="de")]

In [53]:
edm_provided_cho = EDM_ProvidedCHO(
        id=Ref(value=cho_id),
        dc_identifier=identifiers,
        dc_title=titles,
        dc_type=object_types,
        dc_subject=subjects,
        dcterms_medium=media,
        edm_type=Lit(value="3D")
)

### Validation Logic

#### Europeana Logic

In [54]:
try:
    EDM_ProvidedCHO(
            id=Ref(value=cho_id),
            dc_identifier=identifiers,
            dc_title=titles,
            # dc_type=object_types,
            # dc_subject=subjects,
            # dcterms_medium=media,
            edm_type=Lit(value="3D")
    )
except ValidationError as err:
    print(err)

1 validation error for EDM_ProvidedCHO
  Assertion failed, ProvidedCHO must have one of [dc_type, dc_subject, dcterms_termporal, dctermrs_spatial], got self.dc_type=None, self.dc_subject=None, self.dcterms_spatial=None, self.dcterms_temporal=None. [type=assertion_error, input_value={'id': Ref(value='cd2880d...=None, normalize=False)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/assertion_error


#### Kulturpool Logic

##### Explicit

In [55]:
try:
    EDM_ProvidedCHO(
            id=Ref(value=cho_id),
            # dc_identifier=identifiers,
            dc_title=titles,
            dc_type=object_types,
            dc_subject=subjects,
            dcterms_medium=media,
            edm_type=Lit(value="3D")
    )
except ValidationError as err:
    print(err)

1 validation error for EDM_ProvidedCHO
dc_identifier
  Field required [type=missing, input_value={'id': Ref(value='cd2880d...=None, normalize=False)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing


In [56]:
try:
    ORE_Aggregation(
            id=Ref(value=str(uuid4())),
            edm_aggregatedCHO=Ref(value=cho_id),
            edm_dataProvider=Lit(value="Some Data Provider"),
            # edm_isShownAt=Ref(value="https://objects.some-provider.com/1234"),
            edm_isShownBy=Ref(value="https://sketchfab.com/models/some-model/embed"),
            edm_provider=Lit(value="Kulturpool"),
            edm_rights=Ref(value="https://creativecommons.org/licenses/by/4.0/"),
    )
except ValidationError as err:
    print(err)

1 validation error for ORE_Aggregation
  Assertion failed, Aggregation must have edm_isShownAt, got: None. [type=assertion_error, input_value={'id': Ref(value='dbb9cc1.../by/4.0/', is_ref=True)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/assertion_error


In [57]:
try:
    ORE_Aggregation(
            id=Ref(value=str(uuid4())),
            edm_aggregatedCHO=Ref(value=cho_id),
            edm_dataProvider=Lit(value="Some Data Provider"),
            edm_isShownAt=Ref(value="https://objects.some-prover.com/1234"),
            # edm_isShownBy=Ref(value="https://sketchfab.com/models/some-model/embed"),
            edm_provider=Lit(value="Kulturpool"),
            edm_rights=Ref(value="https://creativecommons.org/licenses/by/4.0/"),
    )
except ValidationError as err:
    print(err)  

1 validation error for ORE_Aggregation
  Assertion failed, Aggregation must have edm_isShownBy, got: None. [type=assertion_error, input_value={'id': Ref(value='28fb9f0.../by/4.0/', is_ref=True)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/assertion_error


##### Silent Normalization / Mitigation

In [58]:
ore_aggregation =  ORE_Aggregation(
    id=Ref(value=str(uuid4())),
    edm_aggregatedCHO=Ref(value=cho_id),
    edm_dataProvider=Lit(value="Some Data Provider"),
    edm_isShownAt=Ref(value="https://objects.some-provider.com/1234"),
    edm_isShownBy=Ref(value="https://sketchfab.com/models/some-model/embed"),
    edm_provider=Lit(value="Kulturpool"),
    edm_rights=Ref(value="https://creativecommons.org/licenses/by/4.0/"),
)

In [59]:
pprint(ore_aggregation.model_dump())

{'dc_rights': None,
 'edm_aggregatedCHO': {'is_ref': True,
                       'value': 'cd2880da-d9b3-46ad-8661-be918aa7d624'},
 'edm_dataProvider': {'datatype': None,
                      'lang': None,
                      'normalize': False,
                      'value': 'Some Data Provider'},
 'edm_hasView': None,
 'edm_intermediateProvider': None,
 'edm_isShownAt': {'is_ref': True,
                   'value': 'https://objects.some-provider.com/1234'},
 'edm_isShownBy': {'is_ref': True,
                   'value': 'https://sketchfab.com/models/some-model/embed'},
 'edm_object': None,
 'edm_provider': {'datatype': None,
                  'lang': None,
                  'normalize': False,
                  'value': 'Kulturpool'},
 'edm_rights': {'is_ref': True,
                'value': 'http://creativecommons.org/licenses/by/4.0/'},
 'edm_ugc': None,
 'id': {'is_ref': True, 'value': 'ae531e04-6877-4957-9657-b6b4804b95f3'}}


- Note the normalization from `https://creativecommons.org/licenses/by/4.0/` to `http://creativecommons.org/licenses/by/4.0/`
- The `model_dump` method is inherited from the Pydantic `BaseModel` class. - Nearly every class in the EDMlib is derived from the Pydantic `BaseModel` class.

### Serialization

In [60]:
edm_provided_cho = EDM_ProvidedCHO(
        id=Ref(value=cho_id),
        dc_identifier=identifiers,
        dc_title=titles,
        dc_type=object_types,
        dc_subject=subjects,
        dcterms_medium=media,
        edm_type=Lit(value="3D")
)

In [61]:
ore_aggregation =  ORE_Aggregation(
    id=Ref(value=str(uuid4())),
    edm_aggregatedCHO=Ref(value=cho_id),
    edm_dataProvider=Lit(value="Some Data Provider"),
    edm_isShownAt=Ref(value="https://objects.some-provider.com/1234"),
    edm_isShownBy=Ref(value="https://sketchfab.com/models/some-model/embed"),
    edm_provider=Lit(value="Kulturpool"),
    edm_rights=Ref(value="https://creativecommons.org/licenses/by/4.0/"),
)

In [62]:
edm_record = EDM_Record(
    provided_cho=edm_provided_cho,
    aggregation=ore_aggregation
)

In [63]:
pprint(edm_record.model_dump())

{'aggregation': {'dc_rights': None,
                 'edm_aggregatedCHO': {'is_ref': True,
                                       'value': 'cd2880da-d9b3-46ad-8661-be918aa7d624'},
                 'edm_dataProvider': {'datatype': None,
                                      'lang': None,
                                      'normalize': False,
                                      'value': 'Some Data Provider'},
                 'edm_hasView': None,
                 'edm_intermediateProvider': None,
                 'edm_isShownAt': {'is_ref': True,
                                   'value': 'https://objects.some-provider.com/1234'},
                 'edm_isShownBy': {'is_ref': True,
                                   'value': 'https://sketchfab.com/models/some-model/embed'},
                 'edm_object': None,
                 'edm_provider': {'datatype': None,
                                  'lang': None,
                                  'normalize': False,
                     

In [64]:
print(edm_record.serialize())

<?xml version="1.0" encoding="utf-8"?>
<rdf:RDF
  xmlns:edm="http://www.europeana.eu/schemas/edm/"
  xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
  xmlns:dc="http://purl.org/dc/elements/1.1/"
  xmlns:ore="http://www.openarchives.org/ore/terms/"
  xmlns:dcterms="http://purl.org/dc/terms/"
>
  <ore:Aggregation rdf:about="bd073112-a674-4277-81d7-bd8ea2bd3df6">
    <edm:aggregatedCHO rdf:resource="cd2880da-d9b3-46ad-8661-be918aa7d624"/>
    <edm:dataProvider>Some Data Provider</edm:dataProvider>
    <edm:provider>Kulturpool</edm:provider>
    <edm:rights rdf:resource="http://creativecommons.org/licenses/by/4.0/"/>
    <edm:isShownAt rdf:resource="https://objects.some-provider.com/1234"/>
    <edm:isShownBy rdf:resource="https://sketchfab.com/models/some-model/embed"/>
  </ore:Aggregation>
  <edm:ProvidedCHO rdf:about="cd2880da-d9b3-46ad-8661-be918aa7d624">
    <edm:type>3D</edm:type>
    <dc:identifier>1234</dc:identifier>
    <dc:subject xml:lang="en">Praehistory</dc:subject>
 

In [65]:
pprint(edm_record.get_framed_json_ld())

{'@context': 'https://api.kulturpool.at/ns/v1/edm.json',
 'aggregatedCHO': {'dcType': ['Decorative Weapon',
                              {'id': 'https://www.wikidata.org/wiki/Q728'}],
                   'edmType': '3D',
                   'id': 'cd2880da-d9b3-46ad-8661-be918aa7d624',
                   'identifier': ['1234'],
                   'medium': [{'@language': 'de', '@value': 'Metall'},
                              {'@language': 'de', '@value': 'Bronze'}],
                   'subject': [{'@language': 'en', '@value': 'Praehistory'},
                               {'id': 'https://www.wikidata.org/wiki/Q198'}],
                   'title': [{'@language': 'en',
                              '@value': 'Bronze Ornamental Hatchet'},
                             {'@language': 'de', '@value': 'Bronzezierbeil'}],
                   'type': 'ProvidedCHO'},
 'dataProvider': 'Some Data Provider',
 'edmRights': 'http://creativecommons.org/licenses/by/4.0/',
 'hasView': [],
 'id': 'bd073112

# Parsing/Validating

## Typical usage within an ETL-pipeline @ Kulturpool for a standardized API with EDM-support ...

In [66]:
from edmlib import EDM_Parser
from lxml import etree

In [67]:
oai_record="""<?xml version="1.0" encoding="UTF-8"?>
<rdf:RDF
    xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    xmlns:dc="http://purl.org/dc/elements/1.1/"
    xmlns:dcterms="http://purl.org/dc/terms/"
    xmlns:edm="http://www.europeana.eu/schemas/edm/"
    xmlns:ore="http://www.openarchives.org/ore/terms/"
    xmlns:foaf="http://xmlns.com/foaf/0.1/">

    <edm:ProvidedCHO rdf:about="http://example.org/item/12345">
        <dc:title>Sunset Over the River</dc:title>
        <dc:creator>Jane Doe</dc:creator>
        <dc:date>1875</dc:date>
        <dc:language>en</dc:language>
        <dc:type>Painting</dc:type>
        <dcterms:spatial>Paris, France</dcterms:spatial>
        <dc:subject>Landscape</dc:subject>
        <dc:description>A romantic depiction of a sunset over the Seine river.</dc:description>
        <edm:type>IMAGE</edm:type>
        <dc:identifier>test 12345</dc:identifier>
    </edm:ProvidedCHO>

    <ore:Aggregation rdf:about="http://example.org/aggregation/12345">
        <edm:aggregatedCHO rdf:resource="http://example.org/item/12345" />
        <edm:isShownAt rdf:resource="http://example.org/view/12345" />
        <edm:isShownBy rdf:resource="http://example.org/media/12345/1.jpg" />
        <edm:hasView rdf:resource="http://example.org/media/12345/2.jpg" />
        <edm:rights rdf:resource="http://rightsstatements.org/vocab/InC/1.0/" />
        <edm:dataProvider>Example Museum</edm:dataProvider>
        <edm:provider>Europeana</edm:provider>
    </ore:Aggregation>
</rdf:RDF>
"""

In [68]:
tree = etree.fromstring(oai_record.encode())

In [69]:
edm_record = EDM_Parser.from_string(etree.tostring(tree)).parse()


In [70]:
pprint(edm_record.model_dump())

{'aggregation': {'dc_rights': None,
                 'edm_aggregatedCHO': {'is_ref': True,
                                       'value': 'http://example.org/item/12345'},
                 'edm_dataProvider': {'datatype': None,
                                      'lang': None,
                                      'normalize': False,
                                      'value': 'Example Museum'},
                 'edm_hasView': [{'is_ref': True,
                                  'value': 'http://example.org/media/12345/2.jpg'}],
                 'edm_intermediateProvider': None,
                 'edm_isShownAt': {'is_ref': True,
                                   'value': 'http://example.org/view/12345'},
                 'edm_isShownBy': {'is_ref': True,
                                   'value': 'http://example.org/media/12345/1.jpg'},
                 'edm_object': None,
                 'edm_provider': {'datatype': None,
                                  'lang': 'de',
       

In [71]:
oai_record_missing_rights="""<?xml version="1.0" encoding="UTF-8"?>
<rdf:RDF
    xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    xmlns:dc="http://purl.org/dc/elements/1.1/"
    xmlns:dcterms="http://purl.org/dc/terms/"
    xmlns:edm="http://www.europeana.eu/schemas/edm/"
    xmlns:ore="http://www.openarchives.org/ore/terms/"
    xmlns:foaf="http://xmlns.com/foaf/0.1/">

    <edm:ProvidedCHO rdf:about="http://example.org/item/12345">
        <dc:title>Sunset Over the River</dc:title>
        <dc:creator>Jane Doe</dc:creator>
        <dc:date>1875</dc:date>
        <dc:language>en</dc:language>
        <dc:type>Painting</dc:type>
        <dcterms:spatial>Paris, France</dcterms:spatial>
        <dc:subject>Landscape</dc:subject>
        <dc:description>A romantic depiction of a sunset over the Seine river.</dc:description>
        <edm:type>IMAGE</edm:type>
        <dc:identifier>test 12345</dc:identifier>
    </edm:ProvidedCHO>

    <ore:Aggregation rdf:about="http://example.org/aggregation/12345">
        <edm:aggregatedCHO rdf:resource="http://example.org/item/12345" />
        <edm:isShownAt rdf:resource="http://example.org/view/12345" />
        <edm:isShownBy rdf:resource="http://example.org/media/12345/1.jpg" />
        <edm:hasView rdf:resource="http://example.org/media/12345/2.jpg" />
        <edm:dataProvider>Example Museum</edm:dataProvider>
        <edm:provider>Europeana</edm:provider>
    </ore:Aggregation>
</rdf:RDF>
"""

In [72]:
tree = etree.fromstring(oai_record_missing_rights.encode())

In [73]:
try:
   EDM_Parser.from_string(etree.tostring(tree)).parse()
except ValidationError as err:
    print(err)

1 validation error for ORE_Aggregation
edm_rights
  Field required [type=missing, input_value={'id': Ref(value='http://...45/1.jpg', is_ref=True)}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
